# Ứng dụng thực tế

## Mục tiêu
Áp dụng cho các tập log sự kiện (event log). Tìm các tổ hợp sự kiện xuất hiện cùng nhau thường xuyên. Thảo luận về khả năng phát hiện lỗi hay hành vi bất thường.


In [4]:
from pathlib import Path
import subprocess

def find_project_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / 'src' / 'main.jl').exists():
            return p
    raise FileNotFoundError('Khong tim thay thu muc du an chua src/main.jl')

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
SRC_DIR = PROJECT_ROOT / 'src'
NOTEBOOK_DIR = PROJECT_ROOT / 'notebooks'
DATA_DIR = NOTEBOOK_DIR / 'data'
OUTPUT_DIR = NOTEBOOK_DIR / 'output_eventlog'
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('PROJECT_ROOT =', PROJECT_ROOT)
print('SRC_DIR      =', SRC_DIR)
print('DATA_DIR     =', DATA_DIR)
print('OUTPUT_DIR   =', OUTPUT_DIR)

PROJECT_ROOT = C:\Users\Huynh Gia Computer\OneDrive\Year 3 sem 2\Data mining\FrequentDiscovery
SRC_DIR      = C:\Users\Huynh Gia Computer\OneDrive\Year 3 sem 2\Data mining\FrequentDiscovery\src
DATA_DIR     = C:\Users\Huynh Gia Computer\OneDrive\Year 3 sem 2\Data mining\FrequentDiscovery\notebooks\data
OUTPUT_DIR   = C:\Users\Huynh Gia Computer\OneDrive\Year 3 sem 2\Data mining\FrequentDiscovery\notebooks\output_eventlog


In [5]:
# Event dictionary
event_map = {
    1:  'AUTH_SUCCESS',
    2:  'AUTH_FAILURE',
    3:  'PRIV_ESCALATION_ATTEMPT',
    4:  'MULTIPLE_404',
    5:  'SUDO_COMMAND',
    6:  'FILE_PERMISSION_DENIED',
    7:  'SERVICE_RESTART',
    8:  'OUTBOUND_CONN_NEW_IP',
    9:  'HIGH_CPU',
    10: 'BACKUP_COMPLETED',
    11: 'CONFIG_CHANGE',
    12: 'DATABASE_ERROR',
}

transactions = [
    [1, 10],
    [1, 10, 7],
    [2, 3, 5, 6],
    [2, 3, 5, 8, 11],
    [1, 10, 9],
    [2, 3, 5, 6, 8],
    [7, 9, 12],
    [2, 3, 5, 11],
    [1, 10, 11],
    [2, 3, 5, 6],
    [7, 9, 12, 6],
    [2, 3, 5, 8],
    [1, 10],
    [2, 3, 5, 6, 11],
    [7, 9, 12],
    [1, 10, 4],
    [2, 3, 5, 6, 8, 11],
    [1, 10, 7],
    [2, 3, 5, 6],
    [7, 9, 12, 11],
]

input_file = DATA_DIR / 'event_log_demo.txt'
with open(input_file, 'w', encoding='utf-8') as f:
    for tr in transactions:
        f.write(' '.join(str(x) for x in sorted(set(tr))) + '\n')

map_file = DATA_DIR / 'event_id_mapping.txt'
with open(map_file, 'w', encoding='utf-8') as f:
    for k in sorted(event_map):
        f.write(f"{k}: {event_map[k]}\n")

print('Da tao input:', input_file)
print('Da tao mapping:', map_file)
print('So transaction:', len(transactions))

Da tao input: C:\Users\Huynh Gia Computer\OneDrive\Year 3 sem 2\Data mining\FrequentDiscovery\notebooks\data\event_log_demo.txt
Da tao mapping: C:\Users\Huynh Gia Computer\OneDrive\Year 3 sem 2\Data mining\FrequentDiscovery\notebooks\data\event_id_mapping.txt
So transaction: 20


## Quy trình chạy thực nghiệm bằng CLI

### Bước 1. Chuẩn bị tham số
- Chọn `JULIA_CMD` phù hợp với máy đang dùng.
- Đặt `MINSUP` theo tỷ lệ trong khoảng `[0, 1]`.

### Bước 2. Chạy các chế độ
- `-a`: chạy từng thuật toán (`classic`, `projection`, `adjacency`).
- `-c`: so sánh hai thuật toán trên cùng input.
- `-ca`: chạy toàn bộ thuật toán trên cùng input.


In [11]:
import shutil
from pathlib import Path
JULIA_CMD = "julia"
MINSUP = 0.30

def resolve_julia_command(preferred: str = "julia"):
    if Path(preferred).exists() or shutil.which(preferred):
        return preferred
    candidates = []
    home_candidate = Path.home() / "AppData" / "Local" / "Programs"

    if home_candidate.exists():
        for folder in home_candidate.glob("Julia-*"):
            exe = folder / "bin" / "julia.exe"
            if exe.exists():
                candidates.append(exe)

    for root in [Path("C:/Program Files"), Path("C:/Program Files (x86)")]:
        if root.exists():
            for folder in root.glob("Julia-*"):
                exe = folder / "bin" / "julia.exe"
                if exe.exists():
                    candidates.append(exe)

    if candidates:
        return str(sorted(candidates)[-1])
    return None



def run_cli(args):
    cmd = [JULIA_CMD, "main.jl", *args]
    print("> " + " ".join(cmd))
    result = subprocess.run(
        cmd,
        cwd=SRC_DIR,
        text=True,
        capture_output=True,
        check=False,
    )

    if result.stdout.strip():
        print(result.stdout)

    if result.stderr.strip():
        print("STDERR:")
        print(result.stderr)

    if result.returncode != 0:
        raise RuntimeError(f"CLI failed with code {result.returncode}")

resolved = resolve_julia_command(JULIA_CMD)
if resolved is None:
    print("Khong tim thay Julia.")

else:
    JULIA_CMD = resolved

    for alg in ["classic", "projection", "adjacency"]:
        run_cli(["-a", alg, str(input_file), str(OUTPUT_DIR), str(MINSUP)])

    run_cli(["-c", "classic", "projection", str(input_file), str(OUTPUT_DIR), str(MINSUP)])
    run_cli(["-ca", str(input_file), str(OUTPUT_DIR), str(MINSUP)])

> C:\Users\Huynh Gia Computer\AppData\Local\Programs\Julia-1.12.6\bin\julia.exe main.jl -a classic C:\Users\Huynh Gia Computer\OneDrive\Year 3 sem 2\Data mining\FrequentDiscovery\notebooks\data\event_log_demo.txt C:\Users\Huynh Gia Computer\OneDrive\Year 3 sem 2\Data mining\FrequentDiscovery\notebooks\output_eventlog 0.3
Algorithm: classic
file: C:\Users\Huynh Gia Computer\OneDrive\Year 3 sem 2\Data mining\FrequentDiscovery\notebooks\data\event_log_demo.txt
Patterns: 20
Runtime (ns): 153830900
Nodes: 26
Trees: 9
Conditional trees: 10
Projections: 0
Minimum support: 0.3

> C:\Users\Huynh Gia Computer\AppData\Local\Programs\Julia-1.12.6\bin\julia.exe main.jl -a projection C:\Users\Huynh Gia Computer\OneDrive\Year 3 sem 2\Data mining\FrequentDiscovery\notebooks\data\event_log_demo.txt C:\Users\Huynh Gia Computer\OneDrive\Year 3 sem 2\Data mining\FrequentDiscovery\notebooks\output_eventlog 0.3
Algorithm: projection
file: C:\Users\Huynh Gia Computer\OneDrive\Year 3 sem 2\Data mining\Frequen

In [13]:
def minsup_label(minsup: float) -> str:
    percent = round(minsup * 100, 2)
    if float(percent).is_integer():
        return f"{int(percent)}%"
    return f"{percent}%"

def parse_output(path: Path):
    patterns = []
    for line in path.read_text(encoding='utf-8').splitlines():
        line = line.strip()
        if not line:
            continue
        left, right = line.split('#SUP:')
        items = [int(x) for x in left.strip().split()] if left.strip() else []
        sup = int(right.strip())
        patterns.append({'items': items, 'support': sup})
    return patterns

def decode_pattern(items):
    return [event_map[i] for i in items]

label = minsup_label(MINSUP)
base = input_file.stem

all_patterns = {}
for alg in ['classic', 'projection', 'adjacency']:
    out_file = OUTPUT_DIR / f'local_{alg}_{base}_{label}.txt'
    if not out_file.exists():
        print(f'[WARN] Chua co file output: {out_file}')
        continue
    patterns = parse_output(out_file)
    all_patterns[alg] = patterns

    print('\n' + '=' * 70)
    print(f'Algorithm: {alg} | so pattern: {len(patterns)}')
    print('=' * 70)

    for p in sorted(patterns, key=lambda x: (-x['support'], -len(x['items']), x['items']))[:12]:
        print(f"support={p['support']:>2} | ids={p['items']} | events={decode_pattern(p['items'])}")


Algorithm: classic | so pattern: 20
support= 9 | ids=[2, 3, 5] | events=['AUTH_FAILURE', 'PRIV_ESCALATION_ATTEMPT', 'SUDO_COMMAND']
support= 9 | ids=[2, 3] | events=['AUTH_FAILURE', 'PRIV_ESCALATION_ATTEMPT']
support= 9 | ids=[2, 5] | events=['AUTH_FAILURE', 'SUDO_COMMAND']
support= 9 | ids=[3, 5] | events=['PRIV_ESCALATION_ATTEMPT', 'SUDO_COMMAND']
support= 9 | ids=[2] | events=['AUTH_FAILURE']
support= 9 | ids=[3] | events=['PRIV_ESCALATION_ATTEMPT']
support= 9 | ids=[5] | events=['SUDO_COMMAND']
support= 7 | ids=[1, 10] | events=['AUTH_SUCCESS', 'BACKUP_COMPLETED']
support= 7 | ids=[1] | events=['AUTH_SUCCESS']
support= 7 | ids=[6] | events=['FILE_PERMISSION_DENIED']
support= 7 | ids=[10] | events=['BACKUP_COMPLETED']
support= 6 | ids=[2, 3, 5, 6] | events=['AUTH_FAILURE', 'PRIV_ESCALATION_ATTEMPT', 'SUDO_COMMAND', 'FILE_PERMISSION_DENIED']

Algorithm: projection | so pattern: 20
support= 9 | ids=[2, 3, 5] | events=['AUTH_FAILURE', 'PRIV_ESCALATION_ATTEMPT', 'SUDO_COMMAND']
support

In [14]:
# Danh sach su kien co tinh chat canh bao
suspicious_ids = {2, 3, 5, 6, 8, 11, 12}

classic_patterns = all_patterns.get('classic', [])
suspicious_patterns = []
for p in classic_patterns:
    overlap = sorted(set(p['items']) & suspicious_ids)
    if len(overlap) >= 2:
        suspicious_patterns.append((p, overlap))

suspicious_patterns.sort(key=lambda x: (-x[0]['support'], -len(x[0]['items']), x[0]['items']))

print('Top suspicious frequent combinations (classic):')
for p, overlap in suspicious_patterns[:10]:
    print(
        f"support={p['support']:>2} | pattern={decode_pattern(p['items'])} | suspicious_part={decode_pattern(overlap)}"
    )

print('\nInterpretation:')
print('- Neu combo AUTH_FAILURE + PRIV_ESCALATION_ATTEMPT + SUDO_COMMAND xuat hien lap lai, co the la mau tan cong leo thang dac quyen.')
print('- Neu FILE_PERMISSION_DENIED / OUTBOUND_CONN_NEW_IP / CONFIG_CHANGE di cung cac su kien tren, can uu tien dieu tra.')
print('- Frequent itemset giup tim mau lap lai; de bat anomaly hiem, can giam minsup hoac ket hop detector theo thoi gian.')

Top suspicious frequent combinations (classic):
support= 9 | pattern=['AUTH_FAILURE', 'PRIV_ESCALATION_ATTEMPT', 'SUDO_COMMAND'] | suspicious_part=['AUTH_FAILURE', 'PRIV_ESCALATION_ATTEMPT', 'SUDO_COMMAND']
support= 9 | pattern=['AUTH_FAILURE', 'PRIV_ESCALATION_ATTEMPT'] | suspicious_part=['AUTH_FAILURE', 'PRIV_ESCALATION_ATTEMPT']
support= 9 | pattern=['AUTH_FAILURE', 'SUDO_COMMAND'] | suspicious_part=['AUTH_FAILURE', 'SUDO_COMMAND']
support= 9 | pattern=['PRIV_ESCALATION_ATTEMPT', 'SUDO_COMMAND'] | suspicious_part=['PRIV_ESCALATION_ATTEMPT', 'SUDO_COMMAND']
support= 6 | pattern=['AUTH_FAILURE', 'PRIV_ESCALATION_ATTEMPT', 'SUDO_COMMAND', 'FILE_PERMISSION_DENIED'] | suspicious_part=['AUTH_FAILURE', 'PRIV_ESCALATION_ATTEMPT', 'SUDO_COMMAND', 'FILE_PERMISSION_DENIED']
support= 6 | pattern=['AUTH_FAILURE', 'PRIV_ESCALATION_ATTEMPT', 'FILE_PERMISSION_DENIED'] | suspicious_part=['AUTH_FAILURE', 'PRIV_ESCALATION_ATTEMPT', 'FILE_PERMISSION_DENIED']
support= 6 | pattern=['AUTH_FAILURE', 'SUDO_

## Kết quả và ý nghĩa

### 1. Mẫu đồng xuất hiện trong log
Frequent itemset cho thấy những nhóm sự kiện lặp lại nhiều lần trong hệ thống. Đây là cơ sở để mô tả hành vi vận hành bình thường hoặc hành vi cần theo dõi.

### 2. Khả năng phát hiện lỗi và rủi ro bảo mật
Các mẫu có support cao chứa những sự kiện như `AUTH_FAILURE`, `PRIV_ESCALATION_ATTEMPT`, `SUDO_COMMAND` có thể phản ánh chuỗi hành vi tấn công hoặc cấu hình sai lặp lại.

### 3. Cách dùng trong giám sát thực tế
Có thể dùng các mẫu frequent làm baseline. Khi một pattern có support tăng mạnh theo thời gian, hệ thống có thể phát cảnh báo sớm để kiểm tra.

### 4. Hạn chế
Anomaly hiếm thường không đủ support để trở thành frequent itemset. Vì vậy cần kết hợp thêm các kỹ thuật khác như giảm `minsup`, phân tích theo cửa sổ thời gian, hoặc dùng thêm luật kết hợp (confidence/lift).
